In [4]:
from dataclasses import dataclass, field
import datetime
from decimal import Decimal
from simulator.engine import SimulationEngine, SimulationContext, Event
from simulator.wealth import (
    CashAccount, EconomicAgent,
    PaymentContract, PaymentObligation, AccountRef, PaymentSystem,
    MonthlySchedule, DailySchedule, 
    FixedAmountRule, DailyInterestAmountRule,
    SimulationRecorder,
    WealthEventKind, SystemModules,
    PaymentResultPayload, PaymentPayload
)
from typing import Any, cast, Protocol

engine = SimulationEngine(
    start_date=datetime.date(2026, 1, 1)
)

### Actors
class BehaviorRule(Protocol):
    def start(self, owner: "Person", ctx: SimulationContext) -> None: ...
    def on_event(self, owner: "Person", event: Event[Any], ctx: SimulationContext) -> None: ...


@dataclass
class SweepOldCashOnSalary:
    salary_description: str = "Monthly salary"
    cash_account: str = "cash"
    savings_account: str = "flexible_cash_fund"

    def start(self, owner: "Person", ctx: SimulationContext) -> None:
        ctx.subscribe(owner.name, WealthEventKind.PAYMENT_SUCCEEDED)

    def on_event(self, owner: "Person", event: Event[Any], ctx: SimulationContext) -> None:
        result = cast(PaymentResultPayload, event.payload)
        if not result.success:
            return

        payment = result.request
        if (
            payment.receiver.module_name != owner.name
            or payment.receiver.account_name != self.cash_account
            or payment.description != self.salary_description
        ):
            return

        cash = owner.get_account(self.cash_account)
        leftover = cash.balance - payment.amount

        if leftover <= 0:
            return

        ctx.emit(
            date=ctx.today,
            source=owner.name,
            target=SystemModules.PAYMENT_SYSTEM,
            kind=WealthEventKind.PAYMENT_REQUIRED,
            payload=PaymentPayload(
                payer=AccountRef(owner.name, self.cash_account),
                receiver=AccountRef(owner.name, self.savings_account),
                amount=leftover,
                description="automatic monthly sweep",
            ),
            priority=200,
        )

@dataclass
class Person(EconomicAgent):
    behaviors: list[BehaviorRule] = field(default_factory=list)

    def start(self, ctx: SimulationContext) -> None:
        for behavior in self.behaviors:
            behavior.start(self, ctx)

    def handle(self, event: Event[Any], ctx: SimulationContext) -> None:
        for behavior in self.behaviors:
            behavior.on_event(self, event, ctx)

alice = Person(
    name="Alice",
    accounts={
        "cash": CashAccount(name="cash", initial_balance=10000),
        "flexible_cash_fund": CashAccount(name="flexible_cash_fund", initial_balance=10000)
    },
    behaviors=[
        SweepOldCashOnSalary(
            salary_description="Monthly salary",
            cash_account="cash",
            savings_account="flexible_cash_fund",
        )
    ],
)

landlord = EconomicAgent(
    name="Landlord",
    accounts={
        "cash": CashAccount(name="cash", initial_balance=0)
    }
)

company = EconomicAgent(
    name="Tesla",
    accounts={
        "cash": CashAccount(name="cash", initial_balance=0, allow_negative=True)
    }
)

bank = EconomicAgent(
    name="Bank",
    accounts={
        "interest_expense": CashAccount(name="interest_expense", initial_balance=0, allow_negative=True)
    }
)

### Contracts
salary = PaymentObligation(
    name="Monthly salary",
    payer=AccountRef(module_name="Tesla", account_name="cash"),
    receiver=AccountRef(module_name="Alice", account_name="cash"),
    schedule=MonthlySchedule(day=25),
    amount_rule=FixedAmountRule(Decimal("3000"))
)

rent = PaymentObligation(
    name="Monthly rent",
    payer=AccountRef(module_name="Alice", account_name="cash"),
    receiver=AccountRef(module_name="Landlord", account_name="cash"),
    schedule=MonthlySchedule(day=1),
    amount_rule=FixedAmountRule(Decimal("1200"))
)

flexible_cash_interest = PaymentObligation(
    name="flexible-cash-interest",
    payer=AccountRef(module_name="Bank", account_name="interest_expense"),
    receiver=AccountRef(module_name="Alice", account_name="flexible_cash_fund"),
    schedule=DailySchedule(keep_weekend=True),
    amount_rule=DailyInterestAmountRule(
        account=AccountRef(module_name="Alice", account_name="flexible_cash_fund"),
        annual_nominal_rate=Decimal(0.0341),
    )
)

employment_contract = PaymentContract(
    name="employment-contract",
    start_date=datetime.date(2026, 1, 1),
    obligations=[salary]
)

tenancy_contract = PaymentContract(
    name="tenancy-contract",
    start_date=datetime.date(2026, 1, 1),
    obligations=[rent]
)

flexible_cash_interest_agreement = PaymentContract(
    name="flexible-cash-agreement",
    start_date=datetime.date(2026, 1, 1),
    obligations=[flexible_cash_interest]
    
)

## Recorder
recorder = SimulationRecorder(
    name="daily-recorder",
    start_date=datetime.date(2026, 1, 1),
    schedule=DailySchedule(),
)

engine.register_many([
    alice, landlord, company, bank, PaymentSystem(), 
    employment_contract, tenancy_contract, flexible_cash_interest_agreement,
    recorder,
])

engine.start()
print(engine.queued_event_count)
engine.run_until( 
    datetime.date(2026, 12, 31)
)

4


In [6]:
from simulator.wealth import EconomicAgentSnapshot

for record in recorder.history_of("Alice"):
    snap = cast(EconomicAgentSnapshot, record.snapshot)
    print(
        record.date,
        snap.accounts["cash"],
        snap.accounts["flexible_cash_fund"],
    )

2026-01-01 10000 10000
2026-01-02 8800 10001.86858043235128532996873
2026-01-03 8800 10002.80300157918071865190414
2026-01-04 8800 10003.73751002398578727157860
2026-01-05 8800 10004.67210577492227466615280
2026-01-06 8800 10005.60678884014672626406571
2026-01-07 8800 10006.54155922781644951621962
2026-01-08 8800 10007.47641694608951396717181
2026-01-09 8800 10008.41136200312475132633291
2026-01-10 8800 10009.34639440708175553917187
2026-01-11 8800 10010.28151416612088285842763
2026-01-12 8800 10011.21672128840325191532739
2026-01-13 8800 10012.15201578209074379081158
2026-01-14 8800 10013.08739765534600208676543
2026-01-15 8800 10014.02286691633243299725727
2026-01-16 8800 10014.95842357321420537978341
2026-01-17 8800 10015.89406763415625082651974
2026-01-18 8800 10016.82979910732426373557992
2026-01-19 8800 10017.76561800088470138228030
2026-01-20 8800 10018.70152432300478399041143
2026-01-21 8800 10019.63751808185249480351629
2026-01-22 8800 10020.57359928559658015617512
2026-01-23 

In [4]:
recorder.history.history("Alice")

(Record(date=datetime.date(2026, 1, 1), module_id=1952209663856, module_name='Alice', snapshot=EconomicAgentSnapshot(accounts={'cash': Decimal('10000'), 'flexible_cash_fund': Decimal('10000')})),
 Record(date=datetime.date(2026, 1, 2), module_id=1952209663856, module_name='Alice', snapshot=EconomicAgentSnapshot(accounts={'cash': Decimal('10000'), 'flexible_cash_fund': Decimal('10000')})),
 Record(date=datetime.date(2026, 1, 3), module_id=1952209663856, module_name='Alice', snapshot=EconomicAgentSnapshot(accounts={'cash': Decimal('10000'), 'flexible_cash_fund': Decimal('10000')})),
 Record(date=datetime.date(2026, 1, 4), module_id=1952209663856, module_name='Alice', snapshot=EconomicAgentSnapshot(accounts={'cash': Decimal('10000'), 'flexible_cash_fund': Decimal('10000')})),
 Record(date=datetime.date(2026, 1, 5), module_id=1952209663856, module_name='Alice', snapshot=EconomicAgentSnapshot(accounts={'cash': Decimal('10000'), 'flexible_cash_fund': Decimal('10000')})),
 Record(date=datetim

In [15]:
bank.net_worth

Decimal('-929.5103715273816665531439670')